這份資料集用來預測信用卡申請是否會被核准。資料共有 16 個欄位，其中前 15 欄為申請者背景、財務與信用相關特徵，最後 1 欄為 Approval_Status，作為預測目標
+ Feature: Gender, Age, Debt, Married, Bank_Customer, Education_Level, Ethnicity, Years_Employed, Prior_Default, Employed, Credit_Score, Drivers_License, Citizen, Zip_Code, Income
+ Target: Approval_Status

In [ ]:
# 匯入套件並載入資料集
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

cc_apps = pd.read_csv("cc_approvals.data", header=None)

觀察資料後發現，缺失值以 ? 表示，因此先將其轉換為空值，再依欄位型態進行填補：數值欄位以平均值補值，類別欄位以眾數補值

In [ ]:
# 將 "?" 轉換為缺失值
cc_apps = cc_apps.replace("?", np.nan)

# 依欄位型態填補缺失值
for col in cc_apps.columns:
    if cc_apps[col].dtype in [np.float64, np.int64]:
        cc_apps[col] = cc_apps[col].fillna(cc_apps[col].mean())
    else:
        cc_apps[col] = cc_apps[col].fillna(cc_apps[col].mode()[0])

將原始資料集拆分為特徵資料 X 與目標資料 y，再進一步切分為 80% 訓練集與 20% 測試集，用於後續模型訓練與評估

In [ ]:
# 分離特徵和目標資料，並生成訓練集和測試集
X = cc_apps.iloc[:, :-1]
y = cc_apps.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

針對類別型特徵進行 One-hot encoding，並透過 align 對齊訓練集與測試集的欄位，確保兩者具有相同的特徵維度。最後將欄位名稱轉為字串型態，以符合後續模型輸入需求

In [ ]:
# 找出類別型欄位
categorical_columns = X.select_dtypes(include=['object', 'category']).columns

# 對訓練集與測試集進行 One-hot encoding
X_train_encoded = pd.get_dummies(X_train, columns=categorical_columns, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_columns, drop_first=True)

# 對齊欄位，避免訓練集與測試集欄位不一致
X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded, join='left', axis=1, fill_value=0
)

# 將欄位名稱轉成字串型態
X_train_encoded.columns = X_train_encoded.columns.astype(str)
X_test_encoded.columns = X_test_encoded.columns.astype(str)

使用 StandardScaler 對特徵進行標準化，降低不同尺度特徵對模型訓練的影響，特別是對距離或邊界敏感的模型更為重要

In [ ]:
# 對訓練集與測試集進行標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

建立多種分類模型作為基準比較，並先以預設參數進行初步訓練與評估

In [ ]:
# 建立多種分類模型並進行初步比較
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC()
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results[name] = accuracy_score(y_test, y_pred)

print(results)

為不同模型設定超參數搜尋範圍，讓後續 GridSearch 能在合理範圍內尋找最佳組合

In [ ]:
# 定義各模型的超參數搜尋範圍
param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["liblinear", "lbfgs"],
        "max_iter": [100, 200, 500]
    },
    "Random Forest": {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 10, 20],
        "max_features": ["sqrt", "log2"]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 11],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    },
    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf", "poly"],
        "gamma": ["scale", "auto"]
    }
}

本專案以 Accuracy 作為主要評估指標，同時透過 5-fold cross validation 取得 CV Score，以觀察模型在不同資料切分下的穩定性

In [ ]:
# 進行 GridSearch 並記錄各模型最佳結果

final_results = []

for name, model in models.items():
    print(f"Running GridSearch for {name}...")
    grid = GridSearchCV(estimator=model, param_grid=param_grids[name], cv=5, scoring="accuracy")
    grid_result = grid.fit(X_train_scaled, y_train)
    
    best_model = grid_result.best_estimator_
    best_score = best_model.score(X_test_scaled, y_test)
    
    final_results.append({
        "Model": name,
        "Best Params": grid_result.best_params_,
        "CV Score": grid_result.best_score_,
        "Test Score": best_score
    })

# 整理成表格方便比較
results_df = pd.DataFrame([
    {
        "Model": result["Model"],
        "CV Score": result["CV Score"],
        "Accuracy": result["Test Score"]
    }
    for result in final_results
])

print(results_df)